# **Fine-tuning T5-Large Model for Scene-to-Caption Generation 🎵**

In this project, we fine-tune a pretrained `T5-large` model from Hugging Face to generate musical captions based on given scene descriptions.  

We use a custom dataset `scene_caption_pairs.json` containing pairs of **scene descriptions** (such as a snowy evening, a festive celebration, etc.) and **musical captions** describing the matching soundscape.

The objective is to teach the model how to convert a **visual or emotional scene** into a **musical caption** that reflects the mood, instruments, tempo, and genre.

This process involves:
- Loading and preparing the dataset
- Preprocessing and tokenizing the inputs and outputs
- Fine-tuning the `T5-large` model
- Evaluating the model on a validation set
- Saving the fine-tuned model for future use


## Setup

We begin by installing all necessary libraries for model fine-tuning and evaluation.

- `transformers`: for model and tokenizer
- `datasets`: for easy dataset management
- `evaluate` and `rouge_score`: for evaluating the generated captions


In [ ]:
!pip install evaluate rouge_score datasets transformers


## Import Libraries

We import the necessary libraries for data handling, model loading, tokenization, and training:

- `json` and `pandas` for data processing
- `datasets.Dataset` for creating a Hugging Face dataset
- `T5Tokenizer` and `T5ForConditionalGeneration` for working with the T5 model
- `TrainingArguments` and `Trainer` for model fine-tuning


In [ ]:
import json
import pandas as pd
from datasets import Dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration, TrainingArguments, Trainer


## Mount Google Drive

We mount Google Drive to save the fine-tuned model after training.  
This ensures that the trained model and tokenizer are safely stored and can be accessed later.


In [ ]:
# STEP 2: Mount Google Drive to save model
from google.colab import drive
drive.mount('/content/drive')

### **Loading and Preparing the Dataset**

We begin by loading the `scene_caption_pairs.json` file, which contains a collection of scene-caption pairs for training. Each entry consists of:

- **Scene**: A description of a visual or emotional scene.  
  Example:  
  `"A group of friends sets out to chase their dreams, energized by determination and unity."`

- **Musical Caption**: A corresponding musical caption that reflects the mood, genre, tempo, instruments, and other attributes with overall feel of the scene.  
  Example for the above scene:  
  `"A lengthy rock and pop fusion piece, this motivational composition is a tapestry of melodies woven by a string ensemble, alto saxophone, electric and overdriven guitars, and choir. Set in A minor and maintaining a steady 4/4 time signature, it unfolds at a moderate tempo, creating an atmosphere of positivity and energy"`

---

### **Steps for Data Preparation**

1. Standardize the key name to `musical_caption` for consistency.
2. Convert the list of dictionaries into a pandas DataFrame for easy manipulation.
3. Convert the DataFrame into a Hugging Face `Dataset` object for efficient handling during model training.

---

The dataset is now organized and ready for training, with each scene directly paired with its corresponding musical caption.


In [ ]:
with open("scene_caption_pairs.json", "r") as f:
    data = json.load(f)

# Adjust key if needed
for row in data:
    if "caption" in row and "musical_caption" not in row:
        row["musical_caption"] = row.pop("caption")

df = pd.DataFrame(data)
dataset = Dataset.from_pandas(df)


### **Formatting the Dataset for Training**

In this step, we format the dataset to create input-output pairs suitable for training the model. The process involves:

1. **Formatting Each Example**:  
   We create an `input_text` field by prefixing each scene description with `"scene: "` and assign the corresponding musical caption to the `target_text` field. This prepares the data for the model to understand the relationship between a scene and its musical caption.

2. **Splitting the Dataset**:  
   After formatting, we split the dataset into training and validation sets using a 90-10 split, where 10% of the data is allocated for validation.

3. **Saving for Evaluation**:  
   We save the validation dataset separately (`formatted_val_dataset`) for later evaluation.

In [ ]:
def format_example(example):
    example["input_text"] = "scene: " + example["scene"]
    example["target_text"] = example["musical_caption"]
    return example

formatted_dataset = dataset.map(format_example)
formatted_dataset = formatted_dataset.train_test_split(test_size=0.1)
train_dataset = formatted_dataset["train"]
val_dataset = formatted_dataset["test"]

# Save for evaluation
formatted_val_dataset = val_dataset


### **Loading the T5 Model and Tokenizer**

In this step, we load the pre-trained T5 model and tokenizer:

1. **Model Selection**:  
   We use the `"t5-large"` variant of the T5 model. T5 (Text-to-Text Transfer Transformer) is a versatile model that can handle various natural language processing tasks by converting them into a text-to-text format.

2. **Tokenizer Initialization**:  
   The `T5Tokenizer` is used to convert input text into tokens that can be fed into the model. It also converts the model’s output back into human-readable text.

3. **Model Initialization**:  
   We load the pre-trained T5 model (`T5ForConditionalGeneration`) which is specifically fine-tuned for conditional generation tasks, making it suitable for generating text based on an input prompt.


In [ ]:
model_name = "t5-large"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)


### **Tokenizing the Dataset**

In this step, we tokenize both the input text (scene descriptions) and the target text (musical captions) for model training:

1. **Tokenization Parameters**:
   - We set a maximum input length (`MAX_INPUT_LENGTH`) of 128 tokens for the `input_text` and a maximum target length (`MAX_TARGET_LENGTH`) of 256 tokens for the `target_text`.
   - **Padding and Truncation**:  
     We apply padding to ensure all sequences have the same length and truncation to ensure that longer sequences are shortened to fit the specified maximum lengths.

2. **Creating the Labels**:
   - We set the `labels` for the model as the tokenized `input_ids` of the target text, which allows the model to learn the relationship between the input scene and the target musical caption during training.

3. **Applying Tokenization**:
   - The `tokenize()` function is applied to both the training and validation datasets. The `batched=True` parameter allows the function to process multiple examples at once, and the `remove_columns` argument removes the original columns (e.g., `input_text` and `target_text`) after tokenization to keep the dataset clean.

In [ ]:
MAX_INPUT_LENGTH = 128
MAX_TARGET_LENGTH = 256

def tokenize(example):
    input_enc = tokenizer(
        example["input_text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_INPUT_LENGTH,
    )
    target_enc = tokenizer(
        example["target_text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_TARGET_LENGTH,
    )
    input_enc["labels"] = target_enc["input_ids"]
    return input_enc

train_dataset = train_dataset.map(tokenize, batched=True, remove_columns=train_dataset.column_names)
val_dataset = val_dataset.map(tokenize, batched=True, remove_columns=val_dataset.column_names)


### **Training the Model**

In this step, we define the training configuration and initiate the model training using the `Trainer` class from the Hugging Face library.

1. **Training Arguments**:
   - `output_dir`: Specifies the directory to save the trained model.
   - `evaluation_strategy`: We set it to `"epoch"` so that the model is evaluated at the end of each epoch.
   - `per_device_train_batch_size` and `per_device_eval_batch_size`: The batch size for training and evaluation, set to 2 for both.
   - `gradient_accumulation_steps`: This allows us to accumulate gradients over multiple steps (set to 2) before performing a backward pass, effectively increasing the batch size.
   - `num_train_epochs`: The number of times the entire dataset is passed through the model, set to 3 epochs.
   - `save_strategy`: We save the model at the end of each epoch.
   - `load_best_model_at_end`: Ensures that the best model (based on evaluation) is loaded at the end of training.
   - `logging_dir`: Directory to store logs for training progress.
   - `logging_steps`: Logs training progress every 10 steps.
   - `report_to`: Set to `"none"` to avoid reporting to any external services.

2. **Trainer Initialization**:
   - The `Trainer` class is initialized with the model, training arguments, and datasets (`train_dataset` and `val_dataset`). It handles the entire training loop, including forward and backward passes, logging, and saving checkpoints.

3. **Training the Model**:
   - The `trainer.train()` function starts the training process.

In [ ]:
training_args = TrainingArguments(
    output_dir="./t5_large_scene_to_caption",
    evaluation_strategy="epoch",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_dir="./logs",
    logging_steps=10,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

trainer.train()


### **Saving the Model and Tokenizer**

Once training is complete, we save the model and tokenizer for future use or deployment. The saved files can be easily loaded later without retraining.

1. **Saving the Model**:  
   The `save_pretrained()` method saves the trained model to the specified directory on Google Drive (`/content/drive/MyDrive/t5_large_scene_to_caption`).

2. **Saving the Tokenizer**:  
   Similarly, the tokenizer is saved using the `save_pretrained()` method to the same directory, ensuring that the tokenizer used during training is available for future tokenization.

3. **Confirmation**:  
   A message is printed to confirm that the model and tokenizer have been saved successfully to Google Drive.

In [ ]:
model.save_pretrained("/content/drive/MyDrive/t5_large_scene_to_caption")
tokenizer.save_pretrained("/content/drive/MyDrive/t5_large_scene_to_caption")
print("✅ Model saved to Google Drive: /MyDrive/t5_large_scene_to_caption")


# Conclusion

In this project, we successfully fine-tuned a pretrained `T5-large` model on a custom scene-caption dataset.

The model learned to map detailed scene descriptions into musical captions, capturing mood, instruments, tempo, and genre.

Through careful preprocessing, fine-tuning, evaluation, and testing, we demonstrated that large language models like T5 can be adapted to specialized creative tasks such as music captioning from visual or emotional scenes.

This fine-tuned model can serve as a foundation for further applications like:
- Scene-based music composition
- Automatic soundtrack generation
- Creative AI-driven media production

Future improvements could include:
- Training on larger and more diverse datasets
- Incorporating more musical attributes (e.g., genre classification, instrument selection)
- Experimenting with other encoder-decoder architectures like FLAN-T5, BART, or T5-XL.

---
